<a href="https://colab.research.google.com/github/jarimso/BotCafe/blob/main/3_ollama_cafe_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Proyecto Final**

##***Integrantes**: Yasmin Johana Garcia - Javier Ricardo Muñoz Castillo*
ChatBot RAG del cafe colombiano En este cuaderno construimos un ChatBot especializado en cafe colombiano. Donde cambiamos la base de conocimientos del ejercicio anterior y crear respuestas contextuales usando LangChain y un modelo servido con Ollama.

### Configuracion de entorno, Si ejecutas el notebook en Google Colab se instalaran automaticamente las dependencias necesarias. En un entorno local asegurate de contar con `langchain`, `langchain-ollama`, `langchain-community`, `langchain-huggingface`, `faiss-cpu`, `sentence-transformers`, `pandas` y `gradio`.



In [20]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = {package.key for package in pkg_resources.working_set}
IN_COLAB = 'google-colab' in installed_packages
IN_COLAB


True

In [21]:
if IN_COLAB:
    print('Instalando dependencias de Python...')
    !pip install -q langchain-ollama langchain-community langchain-huggingface faiss-cpu sentence-transformers gradio jupyterlab


Instalando dependencias de Python...


In [22]:
if IN_COLAB:
    print('Instalando Ollama...')
    !curl -fsSL https://ollama.com/install.sh | sh


Instalando Ollama...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [23]:
if IN_COLAB:
    import subprocess, time, atexit, signal

    global OLLAMA_PROCESS

    if 'OLLAMA_PROCESS' not in globals() or OLLAMA_PROCESS.poll() is not None:
        OLLAMA_PROCESS = subprocess.Popen(['ollama', 'serve'])
        time.sleep(5)
        print(f'Ollama esta activo en el puerto 11434 (PID {OLLAMA_PROCESS.pid})')

        def _stop_ollama():
            if OLLAMA_PROCESS.poll() is None:
                OLLAMA_PROCESS.terminate()
                try:
                    OLLAMA_PROCESS.wait(timeout=5)
                except Exception:
                    OLLAMA_PROCESS.kill()
        atexit.register(_stop_ollama)
    else:
        print(f'Ollama ya estaba ejecutandose (PID {OLLAMA_PROCESS.pid})')


Ollama ya estaba ejecutandose (PID 3619)


### Notas para Google Colab

- Ejecuta las celdas de instalacion para descargar dependencias y Ollama.
- Cuando finalicen veras el mensaje de que el servicio esta activo; manten la sesion en ejecucion para que no se cierre el proceso.
- Si reinicias el runtime vuelve a correr las celdas de instalacion antes de usar el bot.


## Nueva base de conocimientos
Para este bot preparamos una coleccion de textos sobre historia, regiones productoras, metodos de preparacion y turismo del cafe colombiano. Los archivos viven en `data/cafe_colombiano/` y cada documento puede servir como punto de partida para el proceso de chunking.

In [25]:
import requests
import pandas as pd

# URL base para acceder a los archivos .txt en el commit especificado
base_url = "https://raw.githubusercontent.com/jarimso/BotCafe/a2a79a74c9a623ab623dddd6c02f4013c38c7a99/data/cafe_colombiano"

# Lista de archivos que mencionaste
filenames = [
    "historia_cafe.txt",
    "maridajes.txt",
    "metodos_preparacion.txt",
    "regiones_productoras.txt",
    "sostenibilidad.txt",
    "turismo_cafetero.txt"
]

rows = []
for fname in filenames:
    url = f"{base_url}/{fname}"
    resp = requests.get(url)
    if resp.status_code == 200:
        rows.append({
            'archivo': fname,
            'contenido': resp.text
        })
    else:
        print(f"⚠️ Error {resp.status_code} al descargar {url}")

df_docs = pd.DataFrame(rows)
df_docs


,archivo,contenido
0,historia_cafe.txt,Historia del cafe colombiano\n\n- El cafe lleg...
1,maridajes.txt,Maridajes recomendados para cafes colombianos\...
2,metodos_preparacion.txt,Metodos populares de preparacion del cafe colo...
3,regiones_productoras.txt,Regiones productoras destacados del cafe colom...
4,sostenibilidad.txt,Iniciativas de sostenibilidad en el cafe colom...
5,turismo_cafetero.txt,Turismo cafetero en Colombia\n\n- El Paisaje C...


Observamos que los archivos contienen parrafos relativamente compactos. Aun asi aplicaremos un `RecursiveCharacterTextSplitter` para permitir que el vector store se alimente de fragmentos manejables y con solapamiento controlado.

In [26]:
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

documents = [Document(page_content=row['contenido'], metadata={'fuente': row['archivo']}) for row in rows]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=120)
chunks = text_splitter.split_documents(documents)

print(f'Documentos originales: {len(documents)}')
print(f'Chunks generados: {len(chunks)}')
chunks[0]


Documentos originales: 6
Chunks generados: 11


Document(metadata={'fuente': 'historia_cafe.txt'}, page_content='Historia del cafe colombiano')

## Vector store con FAISS
Usaremos embeddings multilingues (`sentence-transformers/multilingual-e5-small`) para representar los fragmentos y FAISS para realizar la busqueda semantica. El modelo es ligero y funciona bien en español.

In [27]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name='paraphrase-multilingual-MiniLM-L12-v2')
vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

query = 'Que regiones producen cafe suave en Colombia?'
for idx, doc in enumerate(retriever.get_relevant_documents(query), start=1):
    print(f"Fragmento {idx} ({doc.metadata['fuente']}):\n{doc.page_content[:200]}\n")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Fragmento 1 (maridajes.txt):
Maridajes recomendados para cafes colombianos

- Cafe del Eje Cafetero con pan de yuca o almojabana para balancear la acidez citrica.
- Cafe del Huila con brownie de chocolate oscuro para resaltar not

Fragmento 2 (regiones_productoras.txt):
Regiones productoras destacados del cafe colombiano

1. Eje Cafetero (Caldas, Risaralda, Quindio)
   - Altura promedio de 1.400 metros y clima templado con lluvias regulares.
   - Fincas de pequena es

Fragmento 3 (historia_cafe.txt):
- El cafe llego a Colombia a finales del siglo XVIII a traves de los Jesuitas que viajaban por el Caribe.
- A inicios del siglo XX se consolidaron cooperativas regionales para comercializar el grano y



## Conectando con Ollama
El modelo conversacional se ejecuta via Ollama. Antes de lanzar la siguiente celda debes asegurarte de:
1. Tener Ollama instalado localmente.
2. Haber descargado un modelo adecuado, por ejemplo `ollama pull llama3.2:3b`.
3. Ejecutar `ollama serve` en una terminal independiente.Una vez listo, la siguiente configuracion conectara LangChain con el modelo.

In [28]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model='llama3.2:3b', temperature=0.2)


## Cadena de recuperacion y QA
Construimos una cadena que reformula preguntas usando el historial, recupera contexto relevante y genera una respuesta citando las fuentes. Esto permite conversaciones mas naturales sin perder trazabilidad.

In [29]:
!ollama pull llama3.2:3b

In [30]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.schema import HumanMessage, AIMessage

condense_template = (
    'Utiliza el historial y el contexto recuperado para reformular la pregunta del usuario. '
    'Entrega la version final de la pregunta sin mencionar el proceso de reformulacion.'
)

condense_prompt = ChatPromptTemplate.from_messages([
    ('system', condense_template),
    ('placeholder', '{chat_history}'),
    ('human', '{input}')
])

history_aware_retriever = create_history_aware_retriever(
    llm=llm,
    retriever=retriever,
    prompt=condense_prompt
)

system_template = (
    'Eres TintoBot, un asistente experto en cafe colombiano. '
    'Responde de forma clara y citando las fuentes entre parentesis usando el campo fuente. '
    'Si la pregunta no se relaciona con cafe colombiano, explica brevemente por que no puedes responder.'
    '\n\n'
    '{context}'
)

qa_prompt = ChatPromptTemplate.from_messages([
    ('system', system_template),
    ('placeholder', '{chat_history}'),
    ('human', '{input}')
])

qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)
conversational_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

chat_history = []

sample_question = 'Que experiencias turisticas ofrece el Paisaje Cultural Cafetero?'
response = conversational_chain.invoke({'input': sample_question, 'chat_history': chat_history})
chat_history.extend([HumanMessage(content=sample_question), AIMessage(content=response['answer'])])
print(response['answer'])

El Paisaje Cultural Cafetero es una experiencia turística única que combina la riqueza cultural de las regiones cafeteras con la belleza natural del paisaje. [1]

Algunas de las experiencias turísticas más destacadas que ofrece el Paisaje Cultural Cafetero son:

*   Recorridos por cafetales: Puedes caminar por los cafetales y aprender sobre la producción del café, desde la plantación hasta la cosecha.
*   Catas dirigidas: Disfruta de catas de diferentes variedades de café colombiano, seleccionadas para que puedas apreciar sus características únicas.
*   Talleres de barismo: Aprende a preparar café artesanal y experimenta con diferentes técnicas de extracción y presentación.
*   Cabalgatas: Explora los paisajes naturales circundantes durante una cabalgata, como el Parque Nacional Los Nevados o el Parque Natural del Café.

Además, puedes visitar algunos de los pueblos emblemáticos más coloridos de Colombia, como Salento, Filandia, Jardin y Pijao, cada uno con su propia arquitectura única

## Pruebas rapidas
Ejecuta varias preguntas consecutivas para observar como el modelo mantiene el contexto y ofrece respuestas consistentes.

In [31]:
def consultar_preguntas(preguntas):
    historial = []
    resultados = []
    for pregunta in preguntas:
        reply = conversational_chain.invoke({'input': pregunta, 'chat_history': historial})
        historial.extend([HumanMessage(content=pregunta), AIMessage(content=reply['answer'])])
        resultados.append((pregunta, reply['answer']))
    return resultados

preguntas_demo = [
    'Que caracteristicas diferencian el cafe de Huila y el de Narino?',
    'Recomiendame un maridaje para un cafe del Huila.',
    'Puedes sugerir actividades para un turista que visita Salento?'
]

for pregunta, answer in consultar_preguntas(preguntas_demo):
    print('Usuario:', pregunta)
    print('TintoBot:', answer)
    print('-' * 80)


Usuario: Que caracteristicas diferencian el cafe de Huila y el de Narino?
TintoBot: Según [El Paisaje Cultural Cafetero], la región del Narino se destaca por su producción de café con notas florales y aromas frescos, mientras que el café de la Huila tiene un perfil más complejo y sabores predominantes a durazno, panela y cacao.

En particular, las características diferenciadoras entre el café de Narino y el de Huila son:

* El café de Narino tiene una mayor cantidad de notas florales y aromas frescos, mientras que el café de la Huila tiene un perfil más oscuro y complejo.
* El café de Narino es generalmente más ligero en cuerpo y acidez, mientras que el café de la Huila es más intenso y con un cuerpo medio a pesado.

Es importante destacar que estas características pueden variar dependiendo del productor y del proceso de producción, por lo que no todas las variedades de café de Narino o Huila tendrán exactamente las mismas características.
----------------------------------------------

### Como interpretar la evaluacion manual

- Verifica que las respuestas incluyan referencias entre parentesis con el nombre del archivo fuente.
- Formula preguntas fuera de dominio (por ejemplo, sobre futbol) y confirma que el bot rechaza responder.
- Lanza una conversacion de al menos tres turnos para asegurar que mantiene coherencia entre preguntas dependientes.


## Interfaz con Gradio
Finalmente, montamos una interfaz simple de chat para que cualquier persona pueda interactuar con el bot. El historial interno de LangChain se restaura cuando el usuario limpia la conversacion.

In [32]:
import gradio as gr

lc_history = []
conversational_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

def responder(mensaje, historial):
    reply = conversational_chain.invoke({'input': mensaje, 'chat_history': lc_history})
    lc_history.extend([HumanMessage(content=mensaje), AIMessage(content=reply['answer'])])
    historial.append((mensaje, reply['answer']))
    return '', historial

def limpiar():
    lc_history.clear()
    return []

with gr.Blocks() as demo:
    gr.Markdown('# TintoBot: especialista en cafe colombiano')
    chat = gr.Chatbot()
    prompt = gr.Textbox(label='Pregunta', placeholder='Escribe tu duda sobre cafe colombiano')
    boton = gr.Button('Limpiar historial')

    prompt.submit(responder, [prompt, chat], [prompt, chat])
    boton.click(limpiar, outputs=chat)



In [33]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://26e9932d52e77d5df6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
